# Tools and Functions

In [1]:
// IMPORTS

#include <iostream> 
#include <random>
#include <sstream>
#include <string>
#include <chrono>   

In [2]:
// VALUE PRINT

template <typename T>
void vprint(const T& x) {
    std::cout << x << std::endl;
}

In [3]:
// ARRAY PRINT

template <typename T>
void aprint(const T* array, size_t size) {
    for (size_t i = 0; i < size; ++i) {
        std::cout << array[i] << " ";
    }
    std::cout << std::endl; 
}

In [4]:
// MATRIX PRINT (ONLY FOR INTEGER VALUES)

void mprint(int** matrix, size_t rows, size_t cols) {
    for (size_t i = 0; i < rows; ++i) {
        for (size_t j = 0; j < cols; ++j) {
            std::cout << matrix[i][j] << " ";
        }
        std::cout << std::endl;
    }
    std::cout << std::endl;
}

In [5]:
// RANDOM SETUP
// from 1 to 9

std::random_device rd; // Obtain a random number from hardware
std::mt19937 gen(rd()); // Seed the generator
std::uniform_int_distribution<> int_dis(1, 9);

In [6]:
// RANDOM GEN

int randInt(){
    return int_dis(gen);
}

In [51]:
// TIME

class Timer {
public:
    void start() {
        startTimepoint = std::chrono::high_resolution_clock::now();
    }

    void end(const std::string& unit = "milli") {
        auto endTimepoint = std::chrono::high_resolution_clock::now();

        if (unit == "micro") {
            auto duration = std::chrono::duration_cast<std::chrono::microseconds>(endTimepoint - startTimepoint);
            std::cout << "Time taken by function: " << duration.count() << " microseconds" << std::endl;
        } else if (unit == "milli") {
            auto duration = std::chrono::duration_cast<std::chrono::milliseconds>(endTimepoint - startTimepoint);
            std::cout << std::fixed << std::setprecision(3);
            std::cout << "Time taken by function: " << duration.count() << " milliseconds" << std::endl;
        } else if (unit == "sec") {
            auto duration = std::chrono::duration<double>(endTimepoint - startTimepoint);
            std::cout << std::fixed << std::setprecision(3);
            std::cout << "Time taken by function: " << duration.count() << " seconds" << std::endl;
        } else {
            std::cout << "Invalid time unit. Use 'micro', 'milli', or 'sec'." << std::endl;
        }
    }

private:
    std::chrono::time_point<std::chrono::high_resolution_clock> startTimepoint;
};

Timer timer;


# Cache Basics

Get some information about the CPU on the current system (in that case from the codespace).

In [8]:
! lscpu

Architecture:                       x86_64
CPU op-mode(s):                     32-bit, 64-bit
Byte Order:                         Little Endian
Address sizes:                      48 bits physical, 48 bits virtual
CPU(s):                             2
On-line CPU(s) list:                0,1
Thread(s) per core:                 2
Core(s) per socket:                 1
Socket(s):                          1
NUMA node(s):                       1
Vendor ID:                          AuthenticAMD
CPU family:                         25
Model:                              1
Model name:                         AMD EPYC 7763 64-Core Processor
Stepping:                           1
CPU MHz:                            3243.732
BogoMIPS:                           4890.86
Virtualization:                     AMD-V
Hypervisor vendor:                  Microsoft
Virtualization type:                full
L1d cache:                          32 KiB
L1i cache:                          32 KiB
L2 cache:           

Lets look at the cache by its own.

In [9]:
! lscpu | grep -i "cache"

L1d cache:                          32 KiB
L1i cache:                          32 KiB
L2 cache:                           512 KiB
L3 cache:                           32 MiB


We have 2 CPUs available with 32 KiB of L1 cache (sometimes also known as level 1 cache).

The L1 cache is a type of CPU cache that is closest to the processor cores. It is typically split into two distinct caches: L1i (L1 instruction cache) and L1d (L1 data cache). 

Purpose:

- L1i Cache (Instruction Cache):
    - Stores instructions that the CPU needs to execute.
    - Optimizes the fetching of instructions from memory to the CPU.
    - Improves the performance of the instruction pipeline by reducing the time it takes to fetch instructions.

- L1d Cache (Data Cache):
    - Stores data that the CPU needs to process.
    - Optimizes the fetching of data from memory to the CPU.
    - Enhances the performance of data retrieval operations.The L1 cache, or Level 1 cache, is a type of CPU cache that is closest to the processor cores. It is typically split into two distinct caches: L1i (L1 instruction cache) and L1d (L1 data cache). Here are the differences between them:

We also have 512 KiB of L2 and 32 MiB of L3 cache.

Let look at the cache line size.

In [10]:
! getconf LEVEL1_DCACHE_LINESIZE

64


In [11]:
! getconf LEVEL1_ICACHE_LINESIZE

64


In [12]:
! getconf LEVEL2_CACHE_LINESIZE

64


In [13]:
! getconf LEVEL3_CACHE_LINESIZE

64


The cache line size is 64 bytes.

This means, with a L1 cache size of 32 KiB and a cache line size of 64 bytes we have 500 cache lines.
When storing loading data in the L1 cache, we load a whole cache line, so in that case 64 bytes per load.

# Cpp Basics

Get the size of an integer in byte.

In [14]:
vprint(sizeof(int));

4


This means that we use 4 bytes (**32 bits**) per integer, so $−2^{31}$ to $231^{−1231}−1$ for signed integers.

## Vectors / Arrays with malloc

void* malloc(size_t size);

### How malloc Works

- Requesting Memory: When malloc is called, it requests a block of memory from the heap (**RAM**) of the specified size.
- Allocating Memory: The heap manager in the runtime library tries to find a suitable **contiguous** block of free memory that meets the requested size. If it finds such a block, it marks it as used and returns a **pointer** to the beginning of the block.
- Return Value: If successful, malloc returns a void* pointer to the beginning of the allocated memory block. If it fails (e.g., if there is not enough memory available), it returns NULL.

Lets make an array with 10 integer values.
(Note: We have to keep track of the size and / or type of the array to iterate over it.)

In [15]:
int* arr = (int*)malloc(10 * sizeof(int));

We allocate 10 * 4 bytes with malloc and cast the void pointer to an integer pointer that points to the first element.
This integer pointer must be stored in an integer pointer variable, in that case "arr". 

Lets set the value at positon [0] to 1.

In [16]:
arr[0] = 1;

And print it.

In [17]:
vprint(arr[0]);

1


Lets quickly write a function to fill the array with random integer values.
(The array must be an integer array.)

In [18]:
void fillRandom(int* arr, size_t size) {

    for(int i = 0; i < size; i++){
        arr[i] = randInt();
    }

}

And a function that to fill the array with zeros.

In [19]:
void fillZeros(int* arr, size_t size) {

    for(int i = 0; i < size; i++){
        arr[i] = 0;
    }

}

Lets fill the array with values.

In [20]:
fillRandom(arr, 10);

And print it.

In [21]:
aprint(arr, 10);

7 1 9 1 2 9 1 4 4 5 


## Matrices

We want to create a matrix of size n x m or rows x cols.

In [22]:
size_t rows = 4;
size_t cols = 4; 

A matrix is a set of pointers that are pointing to set of values (vecotrs) each.
Thats why we need to create a set of integer pointers first. 

In [23]:
int** matrix = (int**)malloc(rows * sizeof(int*));

Lets fill our now pointers by allocating the space needed.

In [24]:
for (int i = 0; i < rows; ++i) {
    
    matrix[i] = (int*)malloc(cols * sizeof(int));

}

To fill it, we must create a new function.

(Note: We could use the fillRandom function too.)

In [25]:
void fillMatrixRandom(int** matrix, size_t rows, size_t cols){

    for(int i = 0; i < rows; i++){
        
        for(int j = 0; j < cols; j++){
            
            matrix[i][j] = randInt();

        }
    
    }

}

And fill our matrix.

In [26]:
fillMatrixRandom(matrix, rows, cols);

Now we can print it.

In [27]:
mprint(matrix, rows, cols);

5 7 1 1 
3 6 5 9 
8 6 3 4 
7 7 1 5 



Lets clean everything up and make some functions. 

In [28]:
int* createVector(size_t size){
    int* v = (int*)malloc(size * sizeof(int));
    fillRandom(v, size);
    return v;
}

In [29]:
int* createZeroVector(size_t size){
    int* v = (int*)malloc(size * sizeof(int));
    fillZeros(v, size);
    return v;
}

In [30]:
int** createMatrix(size_t rows, size_t cols){
    int** m = (int**)malloc(rows * sizeof(int*));
    for (size_t i = 0; i < rows; i++) {
        m[i] = createVector(cols);
    }

    return m;
}

In [31]:
int** createZeroMatrix(size_t rows, size_t cols){
    int** m = (int**)malloc(rows * sizeof(int*));
    for (size_t i = 0; i < rows; i++) {
        m[i] = createZeroVector(cols);
    }

    return m;
}

And test the functions.

In [32]:
size_t size = 4;

int** m1 = createMatrix(size, size);
int** m2 = createZeroMatrix(size, size);

mprint(m1, size, size);
mprint(m2, size, size);

1 7 8 9 
9 3 1 9 
6 7 9 3 
3 9 9 7 

0 0 0 0 
0 0 0 0 
0 0 0 0 
0 0 0 0 



## Transpose

Lastly, we write a function to transpose a matrix.

In [33]:
int** transpose(int** in, size_t rows, size_t cols){

    int** out = createMatrix(cols, rows);

    for(size_t row = 0; row < rows; row++){

        for(size_t col = 0; col < cols; col++){
    
            out[col][row] = in[row][col];
            
        }
    }
    
    return out;

}

And test the function.

In [34]:
size_t rows = 5;
size_t cols = 3;

int** in = createMatrix(rows, cols);
mprint(in, rows, cols);

int** trans = transpose(in, rows, cols);

// Note the change from rows / cols to cols / rows !!!
mprint(trans, cols, rows);

1 2 9 
7 4 3 
6 3 6 
2 9 6 
1 2 7 

1 7 6 2 1 
2 4 3 9 2 
9 3 6 6 7 



# Matrix Operations

Lets look in some ways to perform the dot product.

For now, we use vectors or matrices of size: $n * 1024$. We define some values first.

In [35]:
unsigned int ntest = 4;

unsigned int n1 = 1024 * 1;      std::cout << "1024 * 1  = " << n1 << " and n * n = " << n1 * n1 << " values per matrix " << std::endl;
unsigned int n4 = 1024 * 4;      std::cout << "1024 * 4  = " << n4 << " and n * n = " << n4 * n4 << " values per matrix " << std::endl;
unsigned int n8 = 1024 * 8;      std::cout << "1024 * 8  = " << n8 << " and n * n = " << n8 * n8 << " values per matrix " << std::endl;
unsigned int n16 = 1024 * 16;    std::cout << "1024 * 16 = " << n16 << " and n * n = " << n16 * n16 << " values per matrix " << std::endl;
unsigned int n32 = 1024 * 32;    std::cout << "1024 * 32 = " << n32 << " and n * n = " << n32 * n32 << " values per matrix " << std::endl;

// 1024 * 64 is the maximum size for an unsigned integer and cant be represented. So we use 1024 * 63.
unsigned int n63 = 1024 * 63;    std::cout << "1024 * 63 = " << n63 << " and n * n = " << n63 * n63 << " values per matrix " << std::endl;

1024 * 1  = 1024 and n * n = 1048576 values per matrix 
1024 * 4  = 4096 and n * n = 16777216 values per matrix 
1024 * 8  = 8192 and n * n = 67108864 values per matrix 
1024 * 16 = 16384 and n * n = 268435456 values per matrix 
1024 * 32 = 32768 and n * n = 1073741824 values per matrix 
1024 * 63 = 64512 and n * n = 4161798144 values per matrix 


## Serial matrix-vector multiplication

First we define the funciton.

In [36]:
int* smvp(int** m, int* v, size_t size){
    
    int* out = createZeroVector(size);

    for(int row = 0; row < size; row++){
        
        for(int col = 0; col < size; col++){
        
            out[row] += m[row][col] * v[col];
        
        }
    }
    
    return out;
}

And test it with the ntest value. Therefore, we define a matrix and a vecotr.

In [37]:
int n = ntest;

int** m = createMatrix(n, n);
mprint(m, n, n);

int* v = createVector(n);
aprint(v, n);

3 1 6 4 
7 9 1 6 
6 3 4 1 
3 1 2 7 

7 2 1 4 


Now we run the calculations and use the inbuild notebook time tracking.

In [38]:
int* out = smvp(m, v, n);

And print it.

In [39]:
aprint(out, n);

45 92 56 53 


Now we can run the function with diffrent sizes for n.

In [40]:
int n = n16;

int** m = createMatrix(n, n);
int* v = createVector(n);

In [41]:
int* out = smvp(m, v, n);

Sometimes we want to mesure the time more precise, so we use a fuction to mesure the time in diffrent units.

In [52]:
timer.start();
int* out = smvp(m, v, n);
timer.end(); // = timer.end("milli");

Time taken by function: 614 milliseconds


In [53]:
timer.start();
int* out = smvp(m, v, n);
timer.end("sec");

Time taken by function: 0.652 seconds


In [54]:
timer.start();
int* out = smvp(m, v, n);
timer.end("micro");

Time taken by function: 692314 microseconds


# Cleanup

At the end, we must free the allocated memory space.

In [43]:
void freeM(int** matrix, int n) {
    for (int i = 0; i < n; ++i) {
        free(matrix[i]); 
    }
    free(matrix); 
}

In [44]:
// free(arr);
// free(arr1);
// free(arr2);
// freeM(matrix, rows);
// freeM(m1, size);
// freeM(m2, size);
// freeM(out, size);